# Module 5: PySpark - Homework

In [1]:
!python --version

Python 3.10.16


## Question 1

In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/09 15:22:45 WARN Utils: Your hostname, angela-HP-EliteBook-850-G8-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.1.107 instead (on interface wlp0s20f3)
25/03/09 15:22:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/09 15:22:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'3.3.2'

## Question 2

In [4]:
df = spark.read \
    .option("header", "true") \
    .parquet('../data/raw/yellow/2024/yellow_tripdata_2024-10.parquet')

df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 02:30:44|  2024-10-01 02:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [9]:
df = df.repartition(4) 
df.write.parquet("../data/pq/yellow/2024/10/") # when saving, the file gets repartitioned

## Question 3

In [7]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [8]:
df.registerTempTable('yellow_taxis_oct_24') # turn df into a table

In [9]:
spark.sql("""
SELECT 
    MIN(tpep_pickup_datetime) AS first_trip,
    MAX(tpep_pickup_datetime) AS last_trip,
    COUNT(*) trip_count
FROM 
    yellow_taxis_oct_24
WHERE
    date(tpep_pickup_datetime) == '2024-10-15';
""").show()

[Stage 3:============================================>              (6 + 2) / 8]

+-------------------+-------------------+----------+
|         first_trip|          last_trip|trip_count|
+-------------------+-------------------+----------+
|2024-10-15 00:00:00|2024-10-15 23:59:59|    125567|
+-------------------+-------------------+----------+



## Question 4

In [19]:
spark.sql("""
SELECT
    MAX(trip_duration) as max_trip_duration
    FROM
        (SELECT
            TIMESTAMPDIFF(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime) AS trip_duration
        FROM
            yellow_taxis_oct_24);
""").show()

+-----------------+
|max_trip_duration|
+-----------------+
|              162|
+-----------------+



## Question 6

In [20]:
# Joining one large and one small table
df_zones = spark.read.csv("../data/taxi_zone_lookup.csv", header=True)
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [23]:
print(df.count(), len(df.columns))

3833771 19


In [24]:
print(df_zones.count(), len(df_zones.columns))

265 4


In [22]:
df_result = df.join(df_zones, df.PULocationID == df_zones.LocationID)
df_result.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- LocationID: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone

In [25]:
print(df_result.count(), len(df_result.columns)) # Check that rows aren't getting duplicated

3833771 23


In [26]:
df_result.registerTempTable('yellow_taxis_zones_joined') # turn df into a table

In [28]:
spark.sql("""
SELECT
    Zone,
    COUNT(*) AS trip_count
    FROM
        yellow_taxis_zones_joined
    GROUP BY
        Zone
    ORDER BY
        trip_count
    LIMIT 5;
""").show()

+--------------------+----------+
|                Zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
|       Rikers Island|         2|
|       Arden Heights|         2|
|         Jamaica Bay|         3|
| Green-Wood Cemetery|         3|
+--------------------+----------+

